# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** For the Content Decay prediction model, one row represents a single piece of published content (content_hash_id) aggregated over a specific mid-panel month.

**Time Window:** We are using March 2026 (2026-03) as our training panel. We are deliberately leaving the final month (June 2026) untouched as our sealed test set.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import duckdb
from google.colab import userdata
import pandas as pd

# Connecting to the DuckDB and Injecting the Secret Hugging Face token
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Defining the warehouse path
rel = "hf://datasets/FlyRank/internship-warehouse"

# Querying the March 2026 partition to verify the data span and row count
query = f"""
SELECT
    COUNT(*) as total_daily_records,
    COUNT(DISTINCT content_hash_id) as unique_pages,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
From read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

print("Verifying Time Window: March 2026...")
df_window = con.sql(query).df()
display(df_window)


Verifying Time Window: March 2026...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_daily_records,unique_pages,start_date,end_date
0,9841378,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context Fields (Not for Training):**

~ content_hash_id : The unique identifier for the page.

~ report_date : Needed for Time-windowing, but not a feature itself.

**Label (Target Variable):**

~ is_declining : A binary flag (True/False) representing our content decay proxy.

**Feature (The Predictors):**

~ total_clicks : 30-day historical click volume.

~ total_impression : 30-day historical impression volume.

~ average_position : The ranking tier of the page.

~ ctr : Click-through rate.

**Excluded:**

~ client_hash_id : **Why?** We are excluding the client identifier because we want the model to learn universal content decay patterns based on SEO metrics, not memorize which specific clients generally perform worse. Including it risks model bias.




In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata
import pandas as pd

# Re-establishing connection and the variables
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Proving the Grain and Checking for Missing Values
query_verify = f"""
WITH monthly_aggregation AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) as total_clicks,
        SUM(gsc_impressions) as total_impressions,
        AVG(gsc_avg_position) as average_position,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) as ctr
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    COUNT(*) as total_aggregated_rows,
    COUNT(content_hash_id) as non_null_ids,
    SUM(CASE WHEN total_clicks IS NULL THEN 1 ELSE 0 END) as missing_clicks,
    SUM(CASE WHEN total_impressions IS NULL THEN 1 ELSE 0 END) as missing_impressions
FROM monthly_aggregation
"""

print("Verifying Grain and Missing Values...")
df_verify = con.sql(query_verify).df()
display(df_verify)

# Previewing the actual engineered feature table
query_peek = f"""
SELECT
    content_hash_id,
    SUM(gsc_clicks) as total_clicks,
    SUM(gsc_impressions) as total_impressions,
    AVG(gsc_avg_position) as average_position,
    SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) as ctr
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id
LIMIT 5
"""
print("\nPreview of the Engineered Features (1 Row = 1 Page):")
df_peek = con.sql(query_peek).df()
display(df_peek)

Verifying Grain and Missing Values...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_aggregated_rows,non_null_ids,missing_clicks,missing_impressions
0,331437,331437,0.0,0.0



Preview of the Engineered Features (1 Row = 1 Page):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,total_clicks,total_impressions,average_position,ctr
0,content_b7e512995f79d5a6,2.0,1140.0,4.394234,0.001754
1,content_05597932fe4da067,0.0,57.0,2.714744,0.000000
2,content_905aa32a0230694e,0.0,149.0,6.481453,0.000000
3,content_05434271b257bb68,6.0,1421.0,6.320337,0.004222
4,content_d056587ff7faca0c,16.0,2770.0,4.459107,0.005776


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**1. Unbalanced Client History:** Clients onboarded at different times (as seen in dim_clients). We have varying historical depths, meaning long-term, multi-year decay trends might be unobservable for newly onboarded clients.


**2. Limited Engagement Signals:** We are relying strictly on Google Search Console (GSC) metrics. This data cannot tell us anything about post-click user behavior (e.g., bounce rate, time on page), which are strong indicators of content quality.

**3. Blind Spots to External Realities:** This dataset only shows the symptoms of decay. It can never tell us the root cause—such as whether a competitor just published a much better article, if Google pushed a massive core algorithm update, or if the page was accidentally deleted by the webmaster.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.